# Notebook 2: Building the Bayesian Scoring Layer

Part of *Quantifying AI Risk*. Hour 2 of 3.

## Setup

Python standard library plus one import from SciPy for the Beta distribution. SciPy is used because it provides the credible interval calculation in a single function call. The math is simple enough to write by hand, but the library version is cleaner.

In [1]:
import random
import math
from collections import defaultdict
from scipy.stats import beta as beta_dist

random.seed(42)

print("Ready.")

Ready.


## Load the telemetry stream

Notebook 1 wrote its stream to `data/telemetry_stream.jsonl`. This notebook reads from that file, the same way a real production pipeline would. Hour 1 emits and persists. Hour 2 reads and scores. Hour 3 reads and simulates. Each stage writes to a known location so the next stage can pick up cleanly.

If the file does not exist because Notebook 1 has not been run yet, the code below falls back to regenerating the stream in memory using the same random seed and the same emit function. That keeps this notebook runnable on its own for learners who want to skip ahead. The production-style read-from-disk path is what runs first.

In [2]:
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta

INPUT_PATH = Path("../data/telemetry_stream.jsonl") if Path("../data/telemetry_stream.jsonl").exists() else Path("data/telemetry_stream.jsonl")

if INPUT_PATH.exists():
    with open(INPUT_PATH) as f:
        stream = [json.loads(line) for line in f]
    print(f"Loaded {len(stream)} events from {INPUT_PATH.resolve()}")
else:
    # Fallback: regenerate the stream in memory if Notebook 1 has not been run.
    # Same seed, same emit function as Hour 1.
    print(f"No telemetry file found at {INPUT_PATH}. Regenerating in memory.")

    def emit_event_with_scenario(event_num):
        event = {
            "timestamp":    (datetime.now(timezone.utc) + timedelta(minutes=event_num)).isoformat(),
            "model_id":     "loan-approval-v2",
            "applicant_id": f"APP-{event_num:04d}",
            "decision":     random.choice(["APPROVED", "REJECTED"]),
        }
        if event_num < 600:
            event["confidence"]    = round(random.betavariate(8, 2), 3)
            event["latency_ms"]    = round(random.gauss(180, 35), 1)
            event["drift_score"]   = round(random.betavariate(2, 40), 4)
            event["fairness_flag"] = random.random() > 0.03
            event["memory_mb"]     = round(random.gauss(512, 30), 1)
        else:
            event["confidence"]    = round(random.betavariate(3, 3), 3)
            event["latency_ms"]    = round(random.gauss(320, 80), 1)
            event["drift_score"]   = round(random.betavariate(8, 10), 4)
            event["fairness_flag"] = random.random() > 0.15
            event["memory_mb"]     = round(random.gauss(620, 50), 1)
        return event

    random.seed(42)
    stream = [emit_event_with_scenario(i) for i in range(1000)]
    print(f"Regenerated {len(stream)} events.")

Loaded 1000 events from /Users/suneetamodekurty2o/Downloads/quantifying-ai-risk-main/data/telemetry_stream.jsonl


## Bring forward the governance map

Same signal-to-dimension map that was built in Hour 1. The scoring engine uses this map to determine which signal feeds which governance pillar.

In [3]:
GOVERNANCE_MAP = {
    "confidence": {
        "dimension": "Reliability & Robustness",
        "healthy":   0.70,
        "direction": "higher_is_better",
    },
    "latency_ms": {
        "dimension": "Operational Health",
        "healthy":   300,
        "direction": "lower_is_better",
    },
    "drift_score": {
        "dimension": "Data & Model Integrity",
        "healthy":   0.10,
        "direction": "lower_is_better",
    },
    "fairness_flag": {
        "dimension": "Fairness & Societal Impact",
        "healthy":   0.95,
        "direction": "higher_is_better",
    },
    "memory_mb": {
        "dimension": "Resilience & Continuity",
        "healthy":   600,
        "direction": "lower_is_better",
    },
}

print("Map loaded. 5 signals across 5 pillars.")

Map loaded. 5 signals across 5 pillars.


## The skeptical prior

A *prior* is the belief you hold about a system before you have seen any evidence from it. In Bayesian reasoning, you start with a prior, then update it as evidence arrives. The result is called the *posterior*, which is your belief after seeing the evidence.

In Beta distribution terms, the prior is two numbers, called alpha and beta. Alpha counts evidence of trustworthy behavior. Beta counts evidence of untrustworthy behavior. The mean of the distribution is alpha divided by alpha plus beta. So a prior of alpha equal to 2 and beta equal to 8 has a mean of 0.20.

That low starting mean is the design choice. It is called a *skeptical prior*, and it encodes a single principle. A new AI system is not trusted until it has produced evidence of trustworthy behavior over time. This is the zero trust principle applied to AI governance, and it is the most important design choice in this notebook.

Compare the skeptical prior to the naive choice of alpha equal to 1 and beta equal to 1. That is a uniform prior with mean 0.50. A uniform prior says "I have no opinion." In governance work, you should have an opinion. The opinion is that untested systems are untrustworthy. The prior encodes that opinion in math, and every score that follows inherits it.

In [4]:
PRIOR_ALPHA = 2   # evidence of trustworthy behavior, starting small
PRIOR_BETA  = 8   # evidence of untrustworthy behavior, starting larger


def make_pillar_state():
    return {"alpha": PRIOR_ALPHA, "beta": PRIOR_BETA}


# One state per pillar
pillar_state = {dim: make_pillar_state() for dim in set(info["dimension"] for info in GOVERNANCE_MAP.values())}

print("Prior state, one per pillar:\n")
for dim, state in pillar_state.items():
    mean = state["alpha"] / (state["alpha"] + state["beta"])
    print(f"  {dim:32s}  alpha={state['alpha']}  beta={state['beta']}  mean={mean:.2f}")
print("\nEvery pillar starts skeptical. Mean trustworthiness of 0.20.")
print("Telemetry has to earn the score upward.")

Prior state, one per pillar:

  Operational Health                alpha=2  beta=8  mean=0.20
  Fairness & Societal Impact        alpha=2  beta=8  mean=0.20
  Resilience & Continuity           alpha=2  beta=8  mean=0.20
  Data & Model Integrity            alpha=2  beta=8  mean=0.20
  Reliability & Robustness          alpha=2  beta=8  mean=0.20

Every pillar starts skeptical. Mean trustworthiness of 0.20.
Telemetry has to earn the score upward.


## Severity weights

Not every governance failure carries the same weight. A latency spike of fifty milliseconds and a fairness violation are both signals outside their threshold, but they are not the same kind of evidence against trustworthiness. The scoring engine has to reflect that difference, otherwise a thousand harmless latency spikes would dominate a handful of serious fairness violations and the score would be meaningless.

A *severity weight* is a multiplier applied to evidence at the moment of the Bayesian update. When a healthy signal arrives, alpha is incremented by the signal's severity weight. When an unhealthy signal arrives, beta is incremented by the same weight. A signal with a severity of 1.0 contributes one unit of evidence. A signal with a severity of 3.0 contributes three units.

Fairness and drift carry the heaviest weights because regulatory frameworks treat them as the most consequential governance dimensions. Latency and memory carry lighter weights because they are operational signals with less direct impact on fundamental rights. The specific numbers below are illustrative. In a real deployment, severity weights are derived from regulatory scope, business domain, and the organization's risk register. The pattern is what matters.

In [5]:
SEVERITY_WEIGHTS = {
    "confidence":    1.0,   # reliability, moderate weight
    "latency_ms":    0.5,   # operational, lower weight
    "drift_score":   2.0,   # data integrity, high weight
    "fairness_flag": 3.0,   # fairness, highest weight
    "memory_mb":     0.5,   # operational, lower weight
}

print("Severity weights:\n")
for signal, weight in sorted(SEVERITY_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f"  {signal:14s}  {weight}x")
print("\nFairness and drift carry more evidence per event than latency or memory.")

Severity weights:

  fairness_flag   3.0x
  drift_score     2.0x
  confidence      1.0x
  latency_ms      0.5x
  memory_mb       0.5x

Fairness and drift carry more evidence per event than latency or memory.


## The Bayesian update

The *Bayesian update* is the operation that turns a prior into a posterior using new evidence. For Beta distributions paired with binary outcomes, the update rule is one of the cleanest in statistics. If the signal is healthy, alpha increases by the signal's severity weight. If the signal is unhealthy, beta increases by the same weight. That is the entire engine. Statisticians call this pattern the Beta-Binomial update.

Fairness is the exception, just as it was in Hour 1. Fairness is not a per-event signal. It is a rolling rate calculated over a window of recent events. The engine tracks the fairness rate over the last fifty events and classifies that aggregate, instead of classifying each individual fairness flag.

Each governance pillar maintains its own alpha and beta values, which means each pillar has its own posterior. The pillars do not get collapsed into a single number until the very last step.

In [6]:
def is_healthy(signal_name, value, gov_map):
    info = gov_map[signal_name]
    if info["direction"] == "lower_is_better":
        return value <= info["healthy"]
    else:
        return value >= info["healthy"]


def update_pillar(state, healthy, severity):
    if healthy:
        state["alpha"] += severity
    else:
        state["beta"] += severity


# Process the stream event by event, updating pillar states as we go
fairness_window = []
WINDOW = 50

# Snapshot trajectory so we can plot score over time later
trajectory = []

for i, event in enumerate(stream):
    # Per event signals
    for signal_name in ["confidence", "latency_ms", "drift_score", "memory_mb"]:
        value = event[signal_name]
        dim = GOVERNANCE_MAP[signal_name]["dimension"]
        healthy = is_healthy(signal_name, value, GOVERNANCE_MAP)
        update_pillar(pillar_state[dim], healthy, SEVERITY_WEIGHTS[signal_name])

    # Rolling rate signal: fairness
    fairness_window.append(event["fairness_flag"])
    if len(fairness_window) > WINDOW:
        fairness_window.pop(0)

    if len(fairness_window) == WINDOW:
        rate = sum(fairness_window) / WINDOW
        dim = GOVERNANCE_MAP["fairness_flag"]["dimension"]
        healthy = rate >= GOVERNANCE_MAP["fairness_flag"]["healthy"]
        update_pillar(pillar_state[dim], healthy, SEVERITY_WEIGHTS["fairness_flag"])

    # Snapshot every 25 events
    if i % 25 == 0:
        snapshot = {"event": i}
        for dim, state in pillar_state.items():
            snapshot[dim] = state["alpha"] / (state["alpha"] + state["beta"])
        trajectory.append(snapshot)

print("Update complete. Final pillar state:\n")
for dim in sorted(pillar_state.keys()):
    state = pillar_state[dim]
    mean = state["alpha"] / (state["alpha"] + state["beta"])
    print(f"  {dim:32s}  alpha={state['alpha']:7.1f}  beta={state['beta']:6.1f}  mean={mean:.3f}")

Update complete. Final pillar state:

  Data & Model Integrity            alpha= 1102.0  beta= 908.0  mean=0.548
  Fairness & Societal Impact        alpha= 1586.0  beta=1277.0  mean=0.554
  Operational Health                alpha=  375.5  beta= 134.5  mean=0.736
  Reliability & Robustness          alpha=  541.0  beta= 469.0  mean=0.536
  Resilience & Continuity           alpha=  370.5  beta= 139.5  mean=0.726


## The credible interval

A point score by itself is not enough. A pillar with a posterior mean of 0.73 backed by thousands of events is a very different situation from a pillar with a posterior mean of 0.73 backed by twenty events. The first is supported by evidence. The second is provisional. If the score does not communicate which case applies, the score is reporting opinion, not evidence.

A *credible interval* is the range within which the true value is most likely to lie, given the evidence collected so far. The Beta distribution provides this interval directly. The 95% credible interval is the range between the 2.5th and 97.5th percentiles of the distribution. A tight interval means the posterior is supported by a large amount of evidence. A wide interval means the score is still tentative.

Every pillar score in this scoring layer carries its credible interval. The interval is what makes the score defensible, because it tells the reader how much weight to place on the number.

In [7]:
def pillar_score(state, ci=0.95):
    a, b = state["alpha"], state["beta"]
    mean = a / (a + b)
    lower = beta_dist.ppf((1 - ci) / 2, a, b)
    upper = beta_dist.ppf(1 - (1 - ci) / 2, a, b)
    n_observations = a + b - (PRIOR_ALPHA + PRIOR_BETA)
    return mean, lower, upper, n_observations


print("Pillar scores with 95% credible intervals:\n")
print(f"  {'Pillar':32s}  {'Score':>8s}  {'Lower':>8s}  {'Upper':>8s}  {'Evidence':>10s}")
print(f"  {'-'*32}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*10}")
for dim in sorted(pillar_state.keys()):
    mean, lower, upper, n = pillar_score(pillar_state[dim])
    print(f"  {dim:32s}  {mean:>8.3f}  {lower:>8.3f}  {upper:>8.3f}  {n:>10.0f}")

print("\nThe right way to report this to a regulator is not '0.73.'")
print("It is '0.73 with 95% CI of [0.68, 0.78] based on N observations.'")

Pillar scores with 95% credible intervals:

  Pillar                               Score     Lower     Upper    Evidence
  --------------------------------  --------  --------  --------  ----------
  Data & Model Integrity               0.548     0.526     0.570        2000
  Fairness & Societal Impact           0.554     0.536     0.572        2853
  Operational Health                   0.736     0.697     0.774         500
  Reliability & Robustness             0.536     0.505     0.566        1000
  Resilience & Continuity              0.726     0.687     0.764         500

The right way to report this to a regulator is not '0.73.'
It is '0.73 with 95% CI of [0.68, 0.78] based on N observations.'


## The system-level score

Each pillar has its own posterior. The next step is to combine those pillars into a single number that an executive can read at a glance. The combination is done by weighting each pillar's posterior by the pillar's regulatory severity. Fairness and data integrity contribute more to the system score than operational health does, for the same reason they carried higher severity weights at the event level.

There is a specific design choice here that matters. The pillars are combined at the very end, not at the beginning. Each pillar updates independently throughout the stream, and only the final posteriors are aggregated.

The reason is explainability. When the system-level score drops, the architecture preserves the ability to say which pillar drove the drop. A system that collapses the signals into a single average at the start of processing produces one number with no underlying explanation. A system that keeps the pillars separate until the end produces one number and the explanation.

In [8]:
PILLAR_WEIGHTS = {
    "Fairness & Societal Impact":   3.0,
    "Data & Model Integrity":       2.0,
    "Reliability & Robustness":     1.0,
    "Operational Health":           0.5,
    "Resilience & Continuity":      0.5,
}


def system_score(pillar_state, pillar_weights):
    numerator_mean = 0
    numerator_lower = 0
    numerator_upper = 0
    total_weight = sum(pillar_weights.values())

    for dim, weight in pillar_weights.items():
        mean, lower, upper, _ = pillar_score(pillar_state[dim])
        numerator_mean += weight * mean
        numerator_lower += weight * lower
        numerator_upper += weight * upper

    return (
        numerator_mean / total_weight,
        numerator_lower / total_weight,
        numerator_upper / total_weight,
    )


sys_mean, sys_lower, sys_upper = system_score(pillar_state, PILLAR_WEIGHTS)

print(f"System trust score:  {sys_mean:.3f}")
print(f"95% CI:              [{sys_lower:.3f}, {sys_upper:.3f}]")
print(f"Width:               {sys_upper - sys_lower:.3f}")

System trust score:  0.575
95% CI:              [0.551, 0.599]
Width:               0.048


## The executive view

A score between 0 and 1 is fine for engineers, who are used to reading probabilities and confidence intervals. For executives, who need to make a decision quickly, the score is mapped onto color bands. Green, amber, red. Each band corresponds to the decision the reader is expected to make.

Green means the system can be deployed and routinely monitored. Amber means the system is deployed but requires active attention. Red means the system should not be in production in its current state and remediation is required.

The specific band thresholds below are illustrative. Each organization will set its own thresholds based on regulatory scope and business risk tolerance.

In [9]:
def band(score):
    if score >= 0.80: return "GREEN"
    if score >= 0.60: return "AMBER"
    return "RED"


def band_meaning(b):
    return {
        "GREEN": "Deploy and monitor.",
        "AMBER": "Deployed, requires attention.",
        "RED":   "Should not be in production in current state.",
    }[b]


print("Executive view\n")
print(f"  System:  {band(sys_mean):5s}  {sys_mean:.3f}  CI [{sys_lower:.3f}, {sys_upper:.3f}]")
print(f"           {band_meaning(band(sys_mean))}\n")

print("Pillar breakdown:")
for dim in sorted(pillar_state.keys()):
    mean, lower, upper, n = pillar_score(pillar_state[dim])
    print(f"  [{band(mean):5s}]  {dim:32s}  {mean:.3f}  CI [{lower:.3f}, {upper:.3f}]")

print("\nWhen the system score drops, the pillar breakdown tells you which")
print("dimension drove the drop. That is the audit trail a regulator expects.")

Executive view

  System:  RED    0.575  CI [0.551, 0.599]
           Should not be in production in current state.

Pillar breakdown:
  [RED  ]  Data & Model Integrity            0.548  CI [0.526, 0.570]
  [RED  ]  Fairness & Societal Impact        0.554  CI [0.536, 0.572]
  [AMBER]  Operational Health                0.736  CI [0.697, 0.774]
  [RED  ]  Reliability & Robustness          0.536  CI [0.505, 0.566]
  [AMBER]  Resilience & Continuity           0.726  CI [0.687, 0.764]

When the system score drops, the pillar breakdown tells you which
dimension drove the drop. That is the audit trail a regulator expects.


## Watching the posterior evolve

A score at a single moment in time is one snapshot. A score traced over the life of the stream tells a much richer story. The scoring engine recorded each pillar's posterior mean every twenty-five events while processing the stream from Hour 1. Plotting those snapshots shows how each pillar's belief shifted over time, and in particular how the drift injected at event 600 propagated into each pillar's trajectory.

If matplotlib is not available in the environment, the text-based output below tells the same story in tabular form.

In [10]:
# Text based trajectory so this runs without matplotlib
print("Pillar mean trajectory (every 25 events)\n")
print(f"  {'Event':>6s}  " + "  ".join(f"{d[:16]:>16s}" for d in sorted(pillar_state.keys())))
print(f"  {'-'*6}  " + "  ".join(f"{'-'*16}" for _ in pillar_state.keys()))

for snap in trajectory[::4]:  # every 100 events
    row = f"  {snap['event']:>6d}  "
    for dim in sorted(pillar_state.keys()):
        row += f"{snap[dim]:>16.3f}  "
    print(row)

print("\nNotice how every pillar climbs during the healthy phase (events 0-599)")
print("and either stalls or degrades once drift begins at event 600.")
print("Fairness and Data Integrity move most because their severity weights")
print("make them most responsive to the signal degradation.")

Pillar mean trajectory (every 25 events)

   Event  Data & Model Int  Fairness & Socie  Operational Heal  Reliability & Ro  Resilience & Con
  ------  ----------------  ----------------  ----------------  ----------------  ----------------
       0             0.333             0.200             0.238             0.182             0.238  
     100             0.906             0.301             0.868             0.730             0.868  
     200             0.898             0.751             0.928             0.763             0.928  
     300             0.889             0.849             0.947             0.785             0.950  
     400             0.892             0.829             0.960             0.783             0.962  
     500             0.903             0.867             0.967             0.781             0.967  
     600             0.909             0.891             0.971             0.784             0.971  
     700             0.780             0.767         

## Why not a weighted average

The theory section in the slides made a claim. A weighted average cannot distinguish a system with a long healthy history that recently declined from a system that has always been borderline. The Bayesian update can. The proof is worth seeing in numbers, because the difference is what justifies the choice of method.

Consider two systems. System A has five hundred healthy events followed by fifty unhealthy ones. System B has fifty healthy events followed by five hundred unhealthy ones, then a recovery where the most recent fifty are healthy. Both systems have the same final fifty events. Both have similar overall unhealthy event counts. A weighted moving average gives the two systems similar scores. The Bayesian update gives the two systems different scores, because it accumulates evidence across the entire history rather than smoothing it into a window.

The simulation below shows the gap.

In [11]:
def simulate_weighted_avg(observations):
    # Simple exponentially weighted average of healthy rate
    alpha_ewma = 0.1
    rate = 0.5  # start neutral
    for obs in observations:
        rate = alpha_ewma * (1 if obs else 0) + (1 - alpha_ewma) * rate
    return rate


def simulate_bayesian(observations, severity=1.0):
    a, b = PRIOR_ALPHA, PRIOR_BETA
    for obs in observations:
        if obs:
            a += severity
        else:
            b += severity
    return a / (a + b)


# System A: long healthy history, recent stumble
system_a = [True] * 500 + [False] * 50

# System B: short healthy history, long stumble, recent recovery
system_b = [True] * 50 + [False] * 500 + [True] * 50

print("Comparing two systems with similar aggregate statistics\n")
print(f"  System A: 500 healthy, 50 unhealthy (recent stumble after long track record)")
print(f"  System B: 50 healthy, 500 unhealthy, 50 healthy (recovery attempt)\n")

print(f"  {'Method':32s}  {'System A':>10s}  {'System B':>10s}")
print(f"  {'-'*32}  {'-'*10}  {'-'*10}")
print(f"  {'Weighted moving average':32s}  {simulate_weighted_avg(system_a):>10.3f}  {simulate_weighted_avg(system_b):>10.3f}")
print(f"  {'Bayesian (Beta-Binomial)':32s}  {simulate_bayesian(system_a):>10.3f}  {simulate_bayesian(system_b):>10.3f}")

print("\nThe moving average converges toward the recent window regardless of history.")
print("The Bayesian score remembers. System A retains most of its earned trust")
print("despite the recent stumble. System B has to rebuild from accumulated doubt.")
print("That is the behavior we said we wanted.")

Comparing two systems with similar aggregate statistics

  System A: 500 healthy, 50 unhealthy (recent stumble after long track record)
  System B: 50 healthy, 500 unhealthy, 50 healthy (recovery attempt)

  Method                              System A    System B
  --------------------------------  ----------  ----------
  Weighted moving average                0.005       0.995
  Bayesian (Beta-Binomial)               0.896       0.167

The moving average converges toward the recent window regardless of history.
The Bayesian score remembers. System A retains most of its earned trust
despite the recent stumble. System B has to rebuild from accumulated doubt.
That is the behavior we said we wanted.


## Persist the pillar posteriors

The same persistence discipline applies here as in Hour 1. Everything built so far in this notebook lives only in the Jupyter kernel's memory. Notebook 3 will sample from these posteriors many thousands of times to run a Monte Carlo simulation. For that to work, the posteriors have to survive past this kernel.

The pillar state is written to disk as JSON, with one entry per pillar. Each entry carries the alpha and beta of the Beta posterior, along with the system-level weights so that Notebook 3 has everything it needs to reconstruct the scoring without re-running this notebook.

In [12]:
OUTPUT_DIR = Path("../data") if Path("../data").exists() else Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "pillar_posteriors.json"

payload = {
    "pillar_state":     pillar_state,
    "pillar_weights":   PILLAR_WEIGHTS,
    "prior_alpha":      PRIOR_ALPHA,
    "prior_beta":       PRIOR_BETA,
    "n_events":         len(stream),
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Wrote pillar posteriors to {OUTPUT_PATH.resolve()}")
print(f"File size: {OUTPUT_PATH.stat().st_size:,} bytes")
print()
print("Pillars persisted:")
for dim, state in pillar_state.items():
    print(f"  {dim:32s}  alpha={state['alpha']:.1f}  beta={state['beta']:.1f}")
print()
print("Notebook 3 will read from this same path.")

Wrote pillar posteriors to /Users/suneetamodekurty2o/Downloads/quantifying-ai-risk-main/data/pillar_posteriors.json
File size: 700 bytes

Pillars persisted:
  Operational Health                alpha=375.5  beta=134.5
  Fairness & Societal Impact        alpha=1586.0  beta=1277.0
  Resilience & Continuity           alpha=370.5  beta=139.5
  Data & Model Integrity            alpha=1102.0  beta=908.0
  Reliability & Robustness          alpha=541.0  beta=469.0

Notebook 3 will read from this same path.


## Your turn: contradiction detection

One of the strongest uses of a Bayesian governance scoring engine is the ability to detect contradictions. A contradiction in governance terms is a situation where the documents say one thing and the telemetry says another. The organization has a policy on paper that prohibits a particular failure mode, while the production system is producing exactly that failure mode in the field. Without a measurement layer, this contradiction stays hidden.

The mechanism is two evidence streams feeding the same pillar, each updating the posterior with its own severity weight.

What to do:

1. Add a document evidence stream alongside the telemetry stream. For each pillar, simulate a few document evidence events. Policy exists is labeled healthy. Policy missing is labeled unhealthy.
2. Update the same pillar posteriors with both streams. Document evidence updates with lower severity than telemetry, because document existence is weaker evidence of behavior than measured behavior is. A weight of around 0.3 is reasonable.
3. Build a function that detects contradictions. If document evidence is strongly positive (for example, the document alpha divided by document total is greater than 0.8) and telemetry evidence is strongly negative (telemetry alpha divided by telemetry total is less than 0.4), that pillar is flagged as a contradiction.
4. Report which pillars in your simulation are contradictions.

This notebook does not provide a solution. In production, the hardest part of this work is not the math. The hardest part is deciding what counts as a document event versus a telemetry event, and what threshold defines a meaningful contradiction. Practice those judgment calls now, while the stakes are low.

In [13]:
# Your implementation

def emit_document_event(pillar_name):
    # Simulate a document evidence event: policy check, audit finding, etc.
    pass


def update_with_both_streams(pillar_state, telemetry_stream, document_stream):
    # Update pillar state using both streams with different severity weights
    pass


def detect_contradictions(pillar_state_tel, pillar_state_doc, threshold=0.3):
    # Return list of pillars where doc evidence and telemetry evidence disagree
    pass


### Your contradiction report

*This is the cell referenced in the exercise above. Write your report here.*

*A contradiction report is the kind of document that lands on a governance committee's agenda or in a board memo, because it surfaces a discrepancy between what the organization claims to do and what it is measurably doing. Treat the exercise as practice for writing that document.*

*Three to five sentences. Include the following:*

*1. Which pillars in your simulation are flagged as contradictions.*
*2. What is the most likely explanation for the contradiction in each case.*
*3. What action you would propose to the governance committee for at least one of the flagged pillars.*

*This is the kind of written rationale that goes into an ISO 42001 management review record. Treat the exercise accordingly.*

## What this notebook produced

A scoring layer with the following components:

- A skeptical prior that treats untested systems as untrustworthy by default
- Severity weights derived from regulatory priorities, applied at the moment of the Bayesian update
- A Beta-Binomial update that runs per pillar, not per system
- Credible intervals on every score, because a point estimate alone is an opinion
- A weighted aggregation from pillar posteriors to a system-level score that preserves explainability
- A color-coded executive view that maps scores to deployment decisions
- A persisted set of pillar posteriors that Notebook 3 will consume

Hour 3 reads from this set of posteriors. The Monte Carlo engine takes them and turns them into a probability distribution of financial outcomes. The trust score is what the engineering team cares about. The financial number is what the board needs. Both rest on the telemetry layer from Hour 1, and this scoring layer is the bridge between them.

The methodology extends from here. Different industries will require different pillars, different severity weights, and different band thresholds. The architecture stays the same.